<div style="background: linear-gradient(135deg, #1e1b4b, #312e81); color:white; padding:25px; border-radius:10px; 
            text-align:center; font-family:'Segoe UI', sans-serif;">

  <h1 style="margin-bottom:8px;"> WikiArt Image Classification</h1>
  <h3 style="margin-top:0; font-style:italic; font-weight:normal; color:#a5b4fc;">
    Transfer Learning with EfficientNetB3
  </h3>

  <hr style="width:60%; border:1px solid #6366f1; margin:15px auto;">

  <p style="margin:5px 0; font-size:15px;">
    <b>Group Project</b> - Deep Learning (2025/2026)
  </p>
  <p style="margin:0; font-size:13px; color:#c7d2fe;">
    Master in Data Science and Advanced Analytics - Nova Information Management School
  </p>
</div>

<br>

<div style="background-color:#1e293b; color:#e0e7ff; padding:15px 20px; border-left:5px solid #6366f1; 
            border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:14px;">

  <b>Notebook Description</b><br>
  Transfer learning approach using an EfficientNetB3 backbone pretrained on ImageNet,
  fine-tuned for artist classification on the WikiArt dataset.
  The model is evaluated using macro F1 score to account for class imbalance across the 23 artist categories.

</div>

<br>

**<h3>Table of Contents</h3>**
* [1. Environment Setup](#1-environment-setup)
* [2. Model Implementation](#2-model)
* [3. Model Evaluation](#3-eval)


<div id="1-environment-setup" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    1. Setup
  </h2>
</div>

## 1.1 Libraries imports

In [6]:
import os
import sys
import yaml
import tensorflow as tf
import keras
from keras import layers

# import utils functions auto-reload
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath('../src'))

from utils import *

from tensorflow.keras.applications import EfficientNetB3

# Load config (relative to notebooks/)
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

SEED = config['seed']
set_seeds(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
os.path.exists('../src/utils.py')

True

<div id="2-model" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    2. Model Implementation
  </h2>
</div>

## 2.1 Transfer Learning using EfficientNetB3

EfficientNetB3 is chosen over ResNet50 for its superior accuracy-to-parameter ratio (Tan & Le, 2019 — [EfficientNet paper](https://arxiv.org/abs/1905.11946)): ~81.6% top-1 ImageNet accuracy with ~12M parameters, vs ResNet50's ~76% with ~25M.

**Input resolution:** EfficientNetB3's native size is 300×300, but **224×224 is used here** to fit GPU memory at a reasonable batch size. The accuracy drop is small — EfficientNet is robust to moderate resolution changes due to its compound scaling design.

**Classification head** follows the [Keras EfficientNet fine-tuning guide](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/):
- `BatchNormalization` after pooling stabilises feature distributions before the dense layers
- Two dense layers (512 → 256) give the head capacity to learn artist-specific style features
- Dropout 0.4 / 0.3 regularises without excessive information loss
- EfficientNet's internal rescaling handles pixel normalisation — no external `preprocess_input` needed

In [8]:
# 224x224 instead of EfficientNetB3's native 300x300 to fit GPU memory at a reasonable batch size.
# The accuracy tradeoff is minor thanks to EfficientNet's compound scaling robustness.
IMG_SIZE = (224, 224)
BATCH_SIZE = config['batch_size']
NUM_CLASSES = config['num_classes']

# All paths relative to notebooks/
train_dir = config['paths']['train_dir']
val_dir   = config['paths']['val_dir']
test_dir  = config['paths']['test_dir']

train_ds, val_ds, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE,
)

Found 9282 files belonging to 23 classes.


W0000 00:00:1776554117.506229    3273 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1776554117.517375    3273 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1776554117.622896    3273 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5286 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060, pci bus id: 0000:2b:00.0, compute capability: 12.0a


Found 1980 files belonging to 23 classes.
Found 2012 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


In [9]:
# --- Data Augmentation ---
data_augmentation = build_standard_augmentation()

# --- Build Model ---
base_model = EfficientNetB3(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # freeze backbone for phase 1

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)     # training=False keeps BN layers in inference mode while frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)     # stabilise pooled features before dense layers
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="efficientnetb3_transfer")
model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "efficientnetb3_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │         5,911 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,713,862 (44.68 MB)

 Trainable params: 927,255 (3.54 MB)

 Non-trainable params: 10,786,607 (41.15 MB)

In [ ]:
# --- Train (Phase 1) ---
# 3e-4 instead of 1e-3 - mudei a LR, recomendacao do bestie
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")],
)

checkpoint_path = config['models']['efficientnetb3']['checkpoint']
callbacks = build_standard_callbacks(
    checkpoint_path=checkpoint_path,
)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

Epoch 1/20


I0000 00:00:1776554138.038229    3474 cuda_dnn.cc:461] Loaded cuDNN version 90700


290/291 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.2172 - f1_macro: 0.1756 - loss: 2.8584
Epoch 1: val_loss improved from None to 1.76782, saving model to ../models/efficientnetb3_best.keras

Epoch 1: finished saving model to ../models/efficientnetb3_best.keras
291/291 ━━━━━━━━━━━━━━━━━━━━ 40s 98ms/step - accuracy: 0.3073 - f1_macro: 0.2636 - loss: 2.4480 - val_accuracy: 0.5131 - val_f1_macro: 0.4544 - val_loss: 1.7678 - learning_rate: 3.0000e-04
Epoch 2/20
290/291 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.4474 - f1_macro: 0.4019 - loss: 1.8879
Epoch 2: val_loss improved from 1.76782 to 1.49076, saving model to ../models/efficientnetb3_best.keras

Epoch 2: finished saving model to ../models/efficientnetb3_best.keras
291/291 ━━━━━━━━━━━━━━━━━━━━ 27s 92ms/step - accuracy: 0.4647 - f1_macro: 0.4267 - loss: 1.8152 - val_accuracy: 0.5793 - val_f1_macro: 0.5345 - val_loss: 1.4908 - learning_rate: 3.0000e-04
Epoch 3/20
 58/291 ━━━━━━━━━━━━━━━━━━━━ 17s 76ms/step - accuracy: 0.4927 -

In [ ]:
# --- Phase 2: Fine-tuning ---

# Unfreeze the last ~50 layers of EfficientNetB3.
# EfficientNet blocks are deeper than ResNet, so more layers must be unfrozen
# to expose the compound-scaled feature extractors to task-specific gradients.
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Very low LR to avoid disrupting pretrained weights
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.F1Score(average="macro", name="f1_macro")],
)

checkpoint_path = config['models']['efficientnetb3']['checkpoint']
callbacks = build_standard_callbacks(
    checkpoint_path=checkpoint_path,
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

<div id="3-eval" style="background-color:#1e293b; padding:18px; border-radius:6px;">
  <h2 style="margin:0; color:#a5b4fc;">
    3. Model Evaluation
  </h2>
</div>

### Learning Curves

In [ ]:
plot_learning_curves([history_phase1, history_phase2], title="EfficientNetB3 Transfer Learning")

### Test Evaluation

In [ ]:
metrics_efficientnet = evaluate_model(model, test_ds, class_names, "EfficientNetB3 Transfer Learning")

In [ ]:
save_history([history_phase1, history_phase2], config['models']['efficientnetb3']['history'])